# Upscayl บน Google Colab

Notebook นี้ใช้ **backend `upscayl-ncnn` และโมเดลจาก Upscayl ทางการ** โดยตรง (NCNN + Vulkan) พร้อมรองรับการอัปโหลดหลายภาพ, เลือกโมเดล/scale, บันทึกผลลง Google Drive และดาวน์โหลดผลลัพธ์เป็น ZIP

> ต้องเลือก Colab runtime ที่มี **NVIDIA GPU** ก่อนรัน — เซลล์ติดตั้งจะลองเตรียม Vulkan ICD ให้ตรงกับไดรเวอร์และตรวจสอบว่า Vulkan เห็น GPU จริง หาก runtime ปัจจุบันไม่มี NVIDIA Vulkan device จะหยุดพร้อมข้อความแจ้งเตือน ไม่ประมวลผลเงียบ ๆ ด้วย CPU

## วิธีใช้

1. ไปที่ **Runtime → Change runtime type → GPU** แล้วเลือก GPU ที่มีให้ใช้
2. รันทุกเซลล์ตามลำดับ
3. เลือกโมเดล/ขนาดผลลัพธ์ใน **ตั้งค่าการ Upscale**
4. อัปโหลดรูป หรือเปิดใช้ Google Drive ในเซลล์ **ตั้งค่าโฟลเดอร์**
5. ประมวลผลและดาวน์โหลดไฟล์ ZIP จากเซลล์สุดท้าย

ไฟล์ภาพถูกประมวลผลใน Colab runtime ของคุณ; Notebook ดาวน์โหลด CLI และโมเดลที่เลือกจาก GitHub ทางการของ Upscayl


In [ ]:
from pathlib import Path
from google.colab import files

# ตั้งเป็น True ถ้าต้องการใช้ Google Drive เก็บ input/output แบบถาวร
USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKSPACE = Path("/content/drive/MyDrive/Upscayl-Colab")
else:
    WORKSPACE = Path("/content/upscayl-colab")

INPUT_DIR = WORKSPACE / "input"
OUTPUT_DIR = WORKSPACE / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input : {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")
print("ถ้าใช้ Drive ให้วางรูปไว้ใน input หรืออัปโหลดผ่านเซลล์ถัดไปได้เลย")


In [ ]:
# ตั้งค่าการ Upscale — แก้ค่าในเซลล์นี้ก่อนเริ่มประมวลผล
MODEL_LABELS = {
    "upscayl-standard-4x": "Standard — ภาพทั่วไป (แนะนำ)",
    "upscayl-lite-4x": "Lite — โมเดลเล็ก/เร็ว",
    "high-fidelity-4x": "High Fidelity — เน้นรายละเอียด",
    "remacri-4x": "Remacri — รายละเอียดและ texture",
    "ultramix-balanced-4x": "Ultramix Balanced — สมดุล",
    "ultrasharp-4x": "Ultrasharp — เน้นความคม",
    "digital-art-4x": "Digital Art — ภาพวาด/กราฟิก",
}

MODEL_ID = "upscayl-standard-4x"  # เลือก model ID จาก MODEL_LABELS
SCALE_FACTOR = 4                  # 2, 3 หรือ 4 เท่า
OUTPUT_FORMAT = "png"             # png, jpg หรือ webp
TILE_SIZE = 0                     # 0 = อัตโนมัติ; ถ้า VRAM น้อยลอง 64 หรือ 32
GPU_ID = 0                        # Colab ส่วนใหญ่มี GPU หมายเลข 0
TTA = False                       # True = เพิ่ม TTA (ช้าลง แต่บางภาพอาจดีขึ้น)

assert MODEL_ID in MODEL_LABELS, f"ไม่รู้จักโมเดล: {MODEL_ID}"
assert SCALE_FACTOR in (2, 3, 4), "SCALE_FACTOR ต้องเป็น 2, 3 หรือ 4"
assert OUTPUT_FORMAT in ("png", "jpg", "webp"), "OUTPUT_FORMAT ต้องเป็น png, jpg หรือ webp"
assert TILE_SIZE == 0 or TILE_SIZE >= 32, "TILE_SIZE ต้องเป็น 0 หรืออย่างน้อย 32"

RUNTIME_DIR = Path("/content/upscayl-colab-runtime")
BIN_DIR = RUNTIME_DIR / "bin"
BIN_PATH = BIN_DIR / "upscayl-bin"
MODEL_DIR = RUNTIME_DIR / "models"  # ชื่อโฟลเดอร์ models เป็นรูปแบบที่ backend ต้องการ
for folder in (RUNTIME_DIR, BIN_DIR, MODEL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print(f"โมเดล: {MODEL_LABELS[MODEL_ID]} ({MODEL_ID})")
print(f"ขนาด: {SCALE_FACTOR}x | format: {OUTPUT_FORMAT} | tile: {TILE_SIZE}")


In [ ]:
# เตรียม NVIDIA Vulkan และดาวน์โหลด CLI ทางการของ Upscayl
import glob
import hashlib
import json
import os
import re
import shutil
import stat
import subprocess
import urllib.request
import zipfile
from pathlib import Path


def run_command(args, *, check=True, env=None):
    result = subprocess.run(args, text=True, capture_output=True, env=env)
    if check and result.returncode != 0:
        detail = (result.stdout or "") + (result.stderr or "")
        raise RuntimeError(
            f"คำสั่งล้มเหลว: {' '.join(map(str, args))}\n{detail[-5000:]}"
        )
    return result


# อ่านเวอร์ชันไดรเวอร์ เพื่อติดตั้ง NVIDIA Vulkan ICD รุ่นเดียวกับ driver branch
smi = run_command(
    ["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
    check=False,
)
if smi.returncode != 0 or not smi.stdout.strip():
    raise RuntimeError(
        "ไม่พบ NVIDIA GPU ใน runtime นี้ กรุณาเลือก Runtime → Change runtime type → GPU "
        "แล้วรันเซลล์ใหม่"
    )

gpu_line = smi.stdout.strip().splitlines()[0]
parts = [part.strip() for part in gpu_line.split(",", 1)]
driver_version = parts[0]
gpu_name = parts[1] if len(parts) > 1 else "NVIDIA GPU"
major_match = re.match(r"(\d+)", driver_version)
if not major_match:
    raise RuntimeError(f"อ่าน NVIDIA driver version ไม่ได้: {driver_version}")
driver_major = major_match.group(1)
print(f"NVIDIA GPU: {gpu_name} | driver {driver_version}")

apt_update = run_command(["apt-get", "update", "-qq"], check=False)
if apt_update.returncode != 0:
    raise RuntimeError(f"apt-get update ล้มเหลว:\n{apt_update.stderr[-3000:]}")


def apt_package_available(package):
    policy = run_command(["apt-cache", "policy", package], check=False).stdout
    match = re.search(r"Candidate:\s*(\S+)", policy)
    return bool(match and match.group(1) != "(none)")


packages = ["libvulkan1", "vulkan-tools"]
matching_driver_package = f"libnvidia-gl-{driver_major}"
if apt_package_available(matching_driver_package):
    packages.append(matching_driver_package)
else:
    print(f"ไม่พบแพ็กเกจ {matching_driver_package} ใน apt; จะลองใช้ Vulkan ICD ที่มีอยู่ใน runtime")
if apt_package_available("libomp5"):
    packages.append("libomp5")

apt_install = run_command(["apt-get", "install", "-y", "-qq", *packages], check=False)
if apt_install.returncode != 0:
    raise RuntimeError(
        f"ติดตั้ง Vulkan dependencies ไม่สำเร็จ ({', '.join(packages)}):\n"
        f"{apt_install.stdout[-2000:]}\n{apt_install.stderr[-3000:]}"
    )

# บังคับใช้ NVIDIA ICD หากแพ็กเกจติดตั้งไฟล์ JSON ไว้แล้ว
icd_candidates = []
for icd_root in ("/usr/share/vulkan/icd.d", "/etc/vulkan/icd.d"):
    icd_candidates.extend(glob.glob(f"{icd_root}/*nvidia*.json"))
NVIDIA_ICD = None
for candidate in sorted(icd_candidates):
    try:
        with open(candidate, "r", encoding="utf-8") as handle:
            if "ICD" in json.load(handle):
                NVIDIA_ICD = candidate
                break
    except (OSError, json.JSONDecodeError):
        continue
if NVIDIA_ICD:
    os.environ["VK_ICD_FILENAMES"] = NVIDIA_ICD
    os.environ["VK_DRIVER_FILES"] = NVIDIA_ICD
    print(f"NVIDIA Vulkan ICD: {NVIDIA_ICD}")
else:
    print("ไม่พบไฟล์ NVIDIA ICD JSON; จะตรวจสอบ Vulkan device ที่ runtime เปิดให้ใช้")

vulkan = run_command(["vulkaninfo", "--summary"], check=False, env=os.environ.copy())
vulkan_report = (vulkan.stdout or "") + (vulkan.stderr or "")
if vulkan.returncode != 0:
    raise RuntimeError(
        "Vulkan เริ่มทำงานไม่ได้ใน Colab runtime นี้\n"
        f"{vulkan_report[-5000:]}\n"
        "ตรวจว่าเลือก GPU runtime แล้ว restart runtime จากนั้นลองใหม่ "
        "(Colab บาง runtime อาจไม่มี Vulkan support)"
    )

has_nvidia_device = bool(re.search(r"vendorID\s*=\s*0x10de\b", vulkan_report, re.IGNORECASE))
if not has_nvidia_device:
    raise RuntimeError(
        "vulkaninfo ไม่พบ NVIDIA device (vendor ID 0x10de) จึงยังใช้ Upscayl NCNN/Vulkan ไม่ได้\n"
        f"{vulkan_report[:3000]}\n"
        "ลองเลือก GPU runtime ใหม่และ restart; การมี CUDA/nvidia-smi อย่างเดียวไม่ยืนยันว่า Vulkan ใช้งานได้"
    )
print("✅ Vulkan ตรวจพบ NVIDIA GPU")


def download_file(url, destination, expected_sha256=None):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file() and destination.stat().st_size > 0:
        if expected_sha256 is None:
            return destination
        current_hash = hashlib.sha256(destination.read_bytes()).hexdigest()
        if current_hash == expected_sha256:
            return destination
    temporary = destination.with_name(destination.name + ".part")
    request = urllib.request.Request(
        url, headers={"User-Agent": "upscayl-colab-notebook/1.0"}
    )
    try:
        with urllib.request.urlopen(request, timeout=180) as response, temporary.open("wb") as output:
            shutil.copyfileobj(response, output)
        if expected_sha256:
            actual_hash = hashlib.sha256(temporary.read_bytes()).hexdigest()
            if actual_hash.lower() != expected_sha256.lower():
                raise RuntimeError(
                    f"SHA-256 ไม่ตรงสำหรับ {destination.name}: {actual_hash}"
                )
        temporary.replace(destination)
    finally:
        if temporary.exists():
            temporary.unlink()
    return destination


# Pin release เพื่อให้ได้ไบนารีที่ตรวจสอบ SHA-256 ได้
UPSCAYL_CLI_TAG = "20251207-174704"
CLI_URL = (
    "https://github.com/upscayl/upscayl-ncnn/releases/download/"
    f"{UPSCAYL_CLI_TAG}/upscayl-bin-{UPSCAYL_CLI_TAG}-linux.zip"
)
CLI_SHA256 = "a9fab3c770b62f2b7a35d8d6d61eb4e8b3aef79128b665c919d080b85a2292f2"
CLI_ARCHIVE = RUNTIME_DIR / f"upscayl-bin-{UPSCAYL_CLI_TAG}-linux.zip"
download_file(CLI_URL, CLI_ARCHIVE, CLI_SHA256)

extract_dir = RUNTIME_DIR / "upscayl-bin-extracted"
extract_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(CLI_ARCHIVE) as archive:
    root = extract_dir.resolve()
    for member in archive.infolist():
        target = (extract_dir / member.filename).resolve()
        if not target.is_relative_to(root):
            raise RuntimeError(f"พบ path ที่ไม่ปลอดภัยใน ZIP: {member.filename}")
    archive.extractall(extract_dir)

executables = [path for path in extract_dir.rglob("upscayl-bin") if path.is_file()]
if not executables:
    raise RuntimeError("ไม่พบไฟล์ upscayl-bin ใน official Linux release archive")
shutil.copy2(executables[0], BIN_PATH)
BIN_PATH.chmod(BIN_PATH.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

link_check = run_command(["ldd", str(BIN_PATH)], check=False)
missing_libraries = [line.strip() for line in link_check.stdout.splitlines() if "not found" in line]
if missing_libraries:
    raise RuntimeError("ไลบรารีที่ CLI ต้องใช้ยังไม่ครบ:\n" + "\n".join(missing_libraries))

help_check = run_command([str(BIN_PATH), "-h"], check=False, env=os.environ.copy())
if "Usage:" not in ((help_check.stdout or "") + (help_check.stderr or "")):
    print("⚠️ ดาวน์โหลด CLI แล้ว แต่ตรวจข้อความ help ไม่ได้; จะลองทำงานต่อ")

print(f"✅ Upscayl NCNN CLI พร้อมใช้งาน: {BIN_PATH}")


## อัปโหลดรูป

รองรับ JPG, PNG และ WebP; อัปโหลดหลายรูปพร้อมกันได้ หากใช้ Google Drive และวางไฟล์ไว้ในโฟลเดอร์ `input` แล้ว ให้ตั้ง `UPLOAD_FILES = False` ในเซลล์อัปโหลดเพื่อข้ามหน้าต่างเลือกไฟล์


In [ ]:
from google.colab import files

UPLOAD_FILES = True  # ตั้ง False หากมีรูปอยู่ใน input อยู่แล้วและไม่ต้องการเปิดหน้าต่างอัปโหลด
uploaded = files.upload() if UPLOAD_FILES else {}
allowed_extensions = {".jpg", ".jpeg", ".png", ".webp"}
saved_count = 0
for uploaded_name, content in (uploaded or {}).items():
    safe_name = Path(uploaded_name.replace("\\", "/")).name
    if Path(safe_name).suffix.lower() not in allowed_extensions:
        print(f"ข้ามไฟล์ที่ไม่รองรับ: {safe_name}")
        continue
    destination = INPUT_DIR / safe_name
    destination.write_bytes(content)
    saved_count += 1
    print(f"บันทึก: {destination}")

existing = [path for path in INPUT_DIR.rglob("*") if path.is_file() and path.suffix.lower() in allowed_extensions]
print(f"\nมีรูปใน input ทั้งหมด {len(existing)} ไฟล์ (อัปโหลดใหม่ {saved_count} ไฟล์)")


In [ ]:
# ดาวน์โหลดเฉพาะโมเดล Upscayl ที่เลือกไว้ในเซลล์ตั้งค่า
import urllib.request

UPSCAYL_MODELS_REF = "a00d55fee90e0f9435d5eaa86e76700df8199af8"
MODEL_BASE_URL = (
    "https://raw.githubusercontent.com/upscayl/upscayl/"
    f"{UPSCAYL_MODELS_REF}/resources/models"
)

for suffix, minimum_bytes in ((".param", 1_000), (".bin", 100_000)):
    model_file = MODEL_DIR / f"{MODEL_ID}{suffix}"
    if not model_file.is_file() or model_file.stat().st_size < minimum_bytes:
        model_url = f"{MODEL_BASE_URL}/{MODEL_ID}{suffix}"
        print(f"กำลังดาวน์โหลด {model_file.name} ...")
        download_file(model_url, model_file)
    if model_file.stat().st_size < minimum_bytes:
        raise RuntimeError(
            f"ไฟล์โมเดล {model_file.name} เล็กผิดปกติ ({model_file.stat().st_size} bytes) "
            "ตรวจ URL หรือเครือข่ายแล้วลองใหม่"
        )
    print(f"✅ {model_file.name}: {model_file.stat().st_size / (1024 * 1024):.1f} MiB")

print(f"พร้อมใช้โมเดล {MODEL_ID}")


## ประมวลผล

ประมวลผลทุกภาพใน `input` (รวมโฟลเดอร์ย่อย) และเก็บผลใน `output` โดยรักษาโครงสร้างโฟลเดอร์เดิม


In [ ]:
import os
import subprocess

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
input_images = sorted(
    path for path in INPUT_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)
if not input_images:
    raise RuntimeError(f"ไม่พบรูปใน {INPUT_DIR}; อัปโหลดรูปหรือวางไฟล์ใน input ก่อน")

processed = []
failures = []
child_env = os.environ.copy()

for index, source in enumerate(input_images, start=1):
    relative = source.relative_to(INPUT_DIR)
    output_folder = OUTPUT_DIR / relative.parent
    output_folder.mkdir(parents=True, exist_ok=True)
    destination = output_folder / (
        f"{source.stem}_upscayl_{SCALE_FACTOR}x_{MODEL_ID}.{OUTPUT_FORMAT}"
    )

    command = [
        str(BIN_PATH),
        "-i", str(source),
        "-o", str(destination),
        "-s", str(SCALE_FACTOR),
        "-m", str(MODEL_DIR),
        "-n", MODEL_ID,
        "-g", str(GPU_ID),
        "-t", str(TILE_SIZE),
        "-f", OUTPUT_FORMAT,
        "-v",
    ]
    if TTA:
        command.append("-x")

    print(f"[{index}/{len(input_images)}] {relative}")
    result = subprocess.run(
        command,
        text=True,
        capture_output=True,
        env=child_env,
    )
    log = (result.stdout or "") + (result.stderr or "")
    if result.returncode != 0 or not destination.is_file() or destination.stat().st_size == 0:
        failures.append((source, log[-5000:] or f"CLI exit code: {result.returncode}"))
        print(f"❌ ล้มเหลว: {relative}")
        if log:
            print(log[-1500:])
        continue

    processed.append((source, destination))
    print(f"✅ เสร็จ: {destination} ({destination.stat().st_size / (1024 * 1024):.1f} MiB)")

print(f"\nสำเร็จ {len(processed)} / {len(input_images)} ไฟล์")
if failures:
    print(f"มีไฟล์ที่ล้มเหลว {len(failures)} ไฟล์ — ดู log ด้านบนเพื่อแก้ปัญหา")


In [ ]:
# แสดงตัวอย่างผลลัพธ์และสร้าง ZIP สำหรับดาวน์โหลด
from IPython.display import display
from PIL import Image
from zipfile import ZIP_STORED, ZipFile
from google.colab import files

if processed:
    source, result_path = processed[0]
    print(f"ภาพต้นฉบับ: {source.name}")
    with Image.open(source) as image:
        preview = image.copy()
        preview.thumbnail((1200, 1200))
        display(preview)

    print(f"ผลลัพธ์: {result_path.name}")
    with Image.open(result_path) as image:
        preview = image.copy()
        preview.thumbnail((1200, 1200))
        display(preview)

    archive_path = Path("/content/upscayl-results.zip")
    with ZipFile(archive_path, "w", compression=ZIP_STORED) as archive:
        for _, result_path in processed:
            archive.write(result_path, arcname=str(result_path.relative_to(OUTPUT_DIR)))
    print(f"สร้าง ZIP แล้ว: {archive_path} ({archive_path.stat().st_size / (1024 * 1024):.1f} MiB)")
    files.download(str(archive_path))
else:
    print(f"ยังไม่มีผลลัพธ์สำเร็จ ตรวจสอบ log และโฟลเดอร์ {OUTPUT_DIR}")


### หมายเหตุเรื่องแหล่งที่มา

Notebook นี้เป็นตัวเชื่อมสำหรับ Colab ที่ดาวน์โหลด Upscayl NCNN backend และโมเดลจาก upstream ตอนใช้งาน โดยตรึงเวอร์ชัน CLI และตรวจ SHA-256 ของ archive ก่อนเรียกใช้ ดูรายละเอียด upstream/ลิขสิทธิ์ได้ใน [README](README.md)
